In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('loan_approval_data.csv')

In [ ]:
df.head()

In [ ]:
# IT IS BINARY CLASSIFICATION PROBLEM

In [ ]:
#HANDLE MISSING VALUES

In [ ]:
categorical_val = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['number']).columns

In [ ]:
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy='mean')
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])

In [ ]:
df.head()

In [ ]:
cat_imp = SimpleImputer(strategy='most_frequent')
df[categorical_val] = cat_imp.fit_transform(df[categorical_val])

EDA – EXPLORATORY DATA ANALYSIS

In [ ]:
# how balanced our classes are

classes_count = df['Loan_Approved'].value_counts()
plt.pie(classes_count,labels=['No','Yes'],autopct='%1.1f%%')
plt.title('IS LOAN APPROVED OR NOT')

In [ ]:
#ANALYSE CATEGORIES
gender_cnt = df['Gender'].value_counts()
ax = sns.barplot(gender_cnt)
ax.bar_label(ax.containers[0])

In [ ]:
#analyze income
sns.histplot(data = df,x = 'Applicant_Income',bins = 20)

In [ ]:
sns.histplot(data = df,x = 'Coapplicant_Income',bins = 20)

In [ ]:
 # OUTLIER - BOX PLOT
sns.boxplot(data = df,x = 'Loan_Approved',y='Applicant_Income')

In [ ]:
fig , axes = plt.subplots(2,2)

sns.boxplot(ax = axes[0,0],data = df , x = "Loan_Approved",y='Applicant_Income')
sns.boxplot(ax = axes[0,1],data = df , x = "Loan_Approved",y='Credit_Score')
sns.boxplot(ax = axes[1,0],data = df , x = "Loan_Approved",y='DTI_Ratio')
sns.boxplot(ax = axes[1,1],data = df , x = "Loan_Approved",y='Savings')

plt.tight_layout()

In [ ]:
 #Loan Approved vs Credit Score
sns.histplot(data = df , x= 'Credit_Score',hue = 'Loan_Approved',bins = 20,multiple='dodge')

In [ ]:
#Removing Applicant Id 
df = df.drop('Applicant_ID',axis = 1)

#Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder , OneHotEncoder

le = LabelEncoder()
df['Education_Level'] = le.fit_transform(df['Education_Level'])
df['Loan_Approved'] = le.fit_transform(df['Loan_Approved'])

In [ ]:
cols = ['Employment_Status','Marital_Status','Loan_Purpose','Property_Area','Gender','Employer_Category']
ohe = OneHotEncoder(drop = 'first',sparse_output= False,handle_unknown = 'ignore')
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded,columns = ohe.get_feature_names_out(cols),index=df.index)
df = pd.concat([df.drop(columns = cols),encoded_df],axis = 1)

In [ ]:
df.head()

#CORRELATION HEATMAP


In [ ]:
num_cols = df.select_dtypes(include='number')
corr_matrix = num_cols.corr()


plt.figure(figsize=(15,8))
sns.heatmap(corr_matrix,annot=True,fmt='.2f',cmap='viridis')

In [ ]:
num_cols.corr()['Loan_Approved'].sort_values(ascending= False)

In [ ]:
X = df.drop('Loan_Approved',axis = 1)
y = df['Loan_Approved']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
#FEATURE SCALING..........
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

#Train and Evaluate Models 



In [ ]:
# Logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

# Evaluation
print("Logistic Regression Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

In [ ]:
# KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation
print("KNN Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation
print("Naive Bayes Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

#AS of now the best model is naive bayes 
# Now Lets do FEATURE ENGINEERING 

In [ ]:
 
df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2
df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"])

X = df.drop(columns=["Loan_Approved", "Credit_Score", "DTI_Ratio"])
y = df["Loan_Approved"]

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

# Evaluation
print("Logistic Regression Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

In [ ]:
# KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation
print("KNN Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation
print("Naive Bayes Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))